In [ ]:
from model.model import MusicSymbolDetector
from model.dataset import MusicSheetDataSet

dataset = MusicSheetDataSet("../ds2_dense", "train")
print(len(dataset))
dataset.showLabels(0)

In [ ]:
import torch
from common.label import Label
# from processor.section_processor import SectionProcessor
from processor.music_processor import MusicParser
from processor import staff_utils
from model.dataset import crop

import torch
import torch
import cv2
import numpy as np
import json

class TrueDetector:
    def __init__(self, file, dataset) -> None:
        self.image = dataset._load_image(file)
        self.sections = staff_utils.section(self.image)
        self.section_images = []
        for section in self.sections:
            imge = self.image[section.y_min:section.y_max,
                              section.x_min:section.x_max]
            self.section_images.append(imge)

        self.labels = dataset._load_labels(file)
        self.times_called = 0

    def __call__(self, image):
        self.times_called += 1
        for section, section_image in zip(self.sections, self.section_images):
            # find the correct section
            if section_image.shape != image.shape:
                continue
            if np.abs(section_image - image).sum() > 10:
                continue
            _, labels = crop(torch.tensor([[]]), self.labels, section, dataset)
            # print(labels)
            return [Label(
                box.tolist(),
                label.item(),
            ) for box, label in zip(labels["boxes"], labels["labels"])]
        print("SECTION NOT FOUND")

for file in dataset.data['images']:
    file_name = f"../ds2_dense/images/{file['filename']}"
    # print(file_name)
    true_detector = TrueDetector(file, dataset)
    parser = MusicParser(file_name)
    for s in parser.sections:
        s.detector = true_detector
    parser.process()
    # print(true_detector.times_called)
    try:
        with open(f"../ds2_dense/music/{file['filename'][:-4]}.json", "+w") as f:
            json.dump(parser.to_dict(), f)
    except:
        print(f"FAILED: {file['filename'][:-4]}")

In [ ]:
from torchvision import utils
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from model.dataset import MusicSheetDataSet, crop, section
# from processor.section_processor import SectionProcessor
from processor import staff_utils
import json

from common.label import Bbox
import torch
import torch
from torch import nn
from pathlib import Path
from common.music import Note

class MusicSheetProcessorDataset(MusicSheetDataSet):
    def __init__(self, root=None, split='train', transform=None):
        super().__init__(root, split, transform)

        print(self.data)
        
        # load notes
        self.notes = {}
        for image in self.data['images']:
            file = Path(root, 'music', image['filename'])
            json_file = file.with_suffix(".json")
            if json_file.exists():
                self.notes[file.stem] = json.loads(json_file.read_bytes())

    def _get_global_labels(self, labels):
        global_labels = []
        for label in labels:
            
            pass
        boxes = []
        names = []

        global_symbols_prefixes = ['clef', 'timeSig', 'key']

        for annotation in labels:
            for cat_id in annotation['cat_id']:
                if (cat_id == None):
                    continue

                category = self.get_category(cat_id)
                if (category['annotation_set'] != 'deepscores'):
                    continue

                for prefix in global_symbols_prefixes:
                    if (category['name'].startswith(prefix)):
                        box = Bbox(annotation['a_bbox'])
                        names.append(int(cat_id))
                        boxes.append(box.bbox)
                        break
        return {
            'boxes': torch.tensor(boxes),
            'labels': torch.tensor(names),
        }
    
    def _load_notes(self, image):
        notes = []
        for staff in self.notes[image]['staffs']:
            for bar in staff['bars']:
                for note in bar['notes']:
                    notes.append(Note().from_dict(note))
        return notes
    
    def _load_labels(self, image):
        labels = []
        for id in image['ann_ids']:
            label = self.data['annotations'][id]
            labels.append(Label(label['a_bbox'], label['cat_id']))
        return labels

    def __getitem__(self, index):
        # get labels for index
        labels = self._load_labels(self.sampling_sections[index].name)

        labels = crop(labels, self.sections[index], self)
        global_labels = self._get_global_labels(labels)

        labels = {
            'boxes': torch.concat([global_labels['boxes'], labels['boxes']]),
            'labels': torch.concat([global_labels['labels'], labels['labels']]),
        }

        notes = crop(notes, self.sections[index], self)

        return labels, notes

    def label_image(self, index):

        image = self._load_image(self.data['images'])
        sections = section(image)

        staffs = staff_utils.get_staffs(image)
        for staff in staffs:
            staff.parent_music = self

ds = MusicSheetProcessorDataset("../ds2_dense", "train")
ds[0]

In [ ]:

class Processor:
    def __init__(self):
        layer = nn.TransformerEncoderLayer(d_model=512, nhead=8)
        self.transformer = nn.TransformerEncoder(layer, num_layers=8)
        self.embedding = nn.Embedding(num_embeddings=137, embedding_dim=1024)
        self.pos_enc = None

        self.note = nn.Linear(1024, 88)  # standard piano
        self.duration = nn.Linear(1024, 64)  # 1/64th to whole note
    
    # def forward(self, inputs):


    def train(self, dataset):
        dataloader = DataLoader(
            dataset.random(10),
            collate_fn=lambda x: zip(*x),
            shuffle=True
        )

        inputs, target = dataloader[0]


        preds = self.model(inputs)

        # preds 


image = dataset._load_image(dataset.sampling_sections[0].name)
target = dataset._load_target(dataset.sampling_sections[0].name)
image = torch.tensor(image).div(255).unsqueeze(0)


plt.imshow(utils.draw_bounding_boxes(
    image.mul(255).type(torch.uint8), target['boxes'], width=4
).moveaxis(0, 2))